#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Create Flag Parameter

In [0]:
# Check whether the Gold dimension table already exists
if spark.catalog.tableExists("store_catalog.gold.dim_customers"):
    incremental_flag = "1"
else:
    incremental_flag = "0"

In [0]:
print(incremental_flag)

1


#Create Dimension customers

###Fetch Relative Columns

In [0]:
# Load distinct customer records from the Gold layer
df_scr = spark.sql('''
    SELECT
        --ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_surrogate_key,
        ci.customer_id,
        ci.customer_key as customer_number,
        ci.first_name,
        ci.last_name,
        la.country,
        ci.marital_status,
        CASE
            WHEN ci.gender <> 'n/a' THEN ci.gender
            ELSE COALESCE(ca.gender, 'n/a')
        END AS gender,
        ca.birth_date AS birthdate,
        ci.create_date
        --ci.crm_cust_info_surrogate_key,
        --ca.erp_CUST_AZ12_surrogate_key,
        --la.erp_LOC_A101_surrogate_key
    FROM store_catalog.gold.dim_crm_cust_info AS ci
    LEFT JOIN store_catalog.gold.dim_erp_cust_az12 AS ca
        ON ci.customer_key = ca.customer_number 
    LEFT JOIN store_catalog.gold.dim_erp_loc_a101 AS la
        ON ci.customer_key = la.customer_number
''')

In [0]:
df_scr.limit(10).display()

customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date
11065,AW00011065,Jessica,Henderson,United States,Married,Female,1979-04-08,2025-10-07
11082,AW00011082,Angela,Butler,United States,Single,Female,1972-02-01,2025-10-07
11414,AW00011414,Ian,Richardson,United Kingdom,Single,Male,1959-09-15,2025-10-08
12211,AW00012211,Victor,Romero,France,Married,Male,1985-02-23,2025-10-08
12350,AW00012350,Logan,Hernandez,Australia,Married,Male,1984-02-22,2025-10-08
12365,AW00012365,Alex,Evans,United States,Married,Male,1953-02-08,2025-10-08
12386,AW00012386,Katherine,Perry,Germany,Single,Female,1977-10-06,2025-10-08
12528,AW00012528,Devin,Wilson,United States,Single,Male,1964-08-29,2025-10-08
12688,AW00012688,Stacy,Navarro,Australia,Single,Female,1978-02-01,2025-10-12
13011,AW00013011,Anne,Ortega,Australia,Single,Female,1970-11-01,2025-10-12


###dim_customers - Initial and Incremental

In [0]:
# Read existing Gold records or create an empty DataFrame for the initial load
if spark.catalog.tableExists("store_catalog.gold.dim_customers") == True:
    df_sink = spark.sql('''
            SELECT
                customer_surrogate_key,
                customer_id,
                customer_number,
                first_name,
                last_name,
                country,
                marital_status,
                gender,
                birthdate,
                create_date
            FROM
                store_catalog.gold.dim_customers
        ''')
else:
    df_sink = spark.sql('''
            SELECT
                1 as customer_surrogate_key,
                ci.customer_id,
                ci.customer_key as customer_number,
                ci.first_name,
                ci.last_name,
                la.country,
                ci.marital_status,
                CASE
                    WHEN ci.gender <> 'n/a' THEN ci.gender
                    ELSE COALESCE(ca.gender, 'n/a')
                END AS gender,
                ca.birth_date AS birthdate,
                ci.create_date
            FROM store_catalog.gold.dim_crm_cust_info AS ci
            LEFT JOIN store_catalog.gold.dim_erp_cust_az12 AS ca
                ON ci.customer_key = ca.customer_number 
            LEFT JOIN store_catalog.gold.dim_erp_loc_a101 AS la
                ON ci.customer_key = la.customer_number
            WHERE
                1 = 0
        ''')

In [0]:
df_sink.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



###Filtering new records and old records

In [0]:
# Match Gold records with existing Gold records by customer ID
df_filter = df_scr.join(df_sink,df_scr["customer_id"] == df_sink["customer_id"], "left")\
        .select(df_scr["customer_id"],df_scr["customer_number"],df_scr["first_name"],
            df_scr["last_name"],df_scr["country"],df_scr["marital_status"],
            df_scr["gender"],df_scr["birthdate"],df_scr["create_date"],df_sink["customer_surrogate_key"]
    )

In [0]:
df_filter.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



**df_filter_old**

In [0]:
# Separate existing customer records
df_filter_old = df_filter.filter(col("customer_surrogate_key").isNotNull())

**df_filter_new**

In [0]:
# Select new customers that require surrogate keys
df_filter_new = df_filter.filter(col("customer_surrogate_key").isNull())\
    .select(df_scr["customer_id"],df_scr["customer_number"],df_scr["first_name"],
            df_scr["last_name"],df_scr["country"],df_scr["marital_status"],
            df_scr["gender"],df_scr["birthdate"],df_scr["create_date"]
    )

In [0]:
df_filter_new.display()

customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date


###Create Surrogate Key

**Fetch the max Surrogate Key from existing table**

In [0]:
# Retrieve the next available surrogate key
if (incremental_flag == "0"):
    max_value = 1
else:
    max_value_df = spark.sql('''SELECT MAX(customer_surrogate_key) FROM store_catalog.gold.dim_customers''')
    max_value = max_value_df.collect()[0][0]+1

In [0]:
print("incremental_flag:", incremental_flag)
print("max_value:", max_value)

incremental_flag: 1
max_value: 18485


**Create Surrogate Key column and Add the max surrogate key**

In [0]:
# Assign unique surrogate keys to new customer records
df_filter_new = df_filter_new.withColumn("customer_surrogate_key",max_value + monotonically_increasing_id())
df_filter_new.display()

customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date,customer_surrogate_key


###Create Final DF - df_filter_old + df_filter_new

In [0]:
# Combine new customers with existing customer records
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.display()

customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date,customer_surrogate_key
11065,AW00011065,Jessica,Henderson,United States,Married,Female,1979-04-08,2025-10-07,1
11082,AW00011082,Angela,Butler,United States,Single,Female,1972-02-01,2025-10-07,2
11414,AW00011414,Ian,Richardson,United Kingdom,Single,Male,1959-09-15,2025-10-08,3
12211,AW00012211,Victor,Romero,France,Married,Male,1985-02-23,2025-10-08,4
12350,AW00012350,Logan,Hernandez,Australia,Married,Male,1984-02-22,2025-10-08,5
12365,AW00012365,Alex,Evans,United States,Married,Male,1953-02-08,2025-10-08,6
12386,AW00012386,Katherine,Perry,Germany,Single,Female,1977-10-06,2025-10-08,7
12528,AW00012528,Devin,Wilson,United States,Single,Male,1964-08-29,2025-10-08,8
12688,AW00012688,Stacy,Navarro,Australia,Single,Female,1978-02-01,2025-10-12,9
13011,AW00013011,Anne,Ortega,Australia,Single,Female,1970-11-01,2025-10-12,10


#SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("store_catalog.gold.dim_customers"):
    # Incremental Run
    # Merge customer records into the Gold dimension table
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_customers")

    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.customer_surrogate_key = src.customer_surrogate_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    # Initial Run
    # Create the Gold dimension table during the initial load
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_customers")\
        .saveAsTable("store_catalog.gold.dim_customers")

In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_customers 

customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date,customer_surrogate_key
11065,AW00011065,Jessica,Henderson,United States,Married,Female,1979-04-08,2025-10-07,1
11082,AW00011082,Angela,Butler,United States,Single,Female,1972-02-01,2025-10-07,2
11414,AW00011414,Ian,Richardson,United Kingdom,Single,Male,1959-09-15,2025-10-08,3
12211,AW00012211,Victor,Romero,France,Married,Male,1985-02-23,2025-10-08,4
12350,AW00012350,Logan,Hernandez,Australia,Married,Male,1984-02-22,2025-10-08,5
12365,AW00012365,Alex,Evans,United States,Married,Male,1953-02-08,2025-10-08,6
12386,AW00012386,Katherine,Perry,Germany,Single,Female,1977-10-06,2025-10-08,7
12528,AW00012528,Devin,Wilson,United States,Single,Male,1964-08-29,2025-10-08,8
12688,AW00012688,Stacy,Navarro,Australia,Single,Female,1978-02-01,2025-10-12,9
13011,AW00013011,Anne,Ortega,Australia,Single,Female,1970-11-01,2025-10-12,10


In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_customers  ORDER BY customer_surrogate_key DESC LIMIT 10

customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date,customer_surrogate_key
29467,AW00029467,Monica,Mehta,Germany,Married,Female,1972-10-12,2026-01-25,18484
29458,AW00029458,Danny,Moreno,Australia,Single,Male,1982-08-24,2026-01-25,18483
29343,AW00029343,Glenn,Wu,Australia,Married,Male,1945-10-12,2026-01-25,18482
29305,AW00029305,Beth,Alvarez,Australia,Single,Female,1964-11-09,2026-01-25,18481
29252,AW00029252,Alex,Parker,United States,Married,Male,1967-03-08,2026-01-25,18480
29190,AW00029190,Kaitlin,Suri,Australia,Single,Female,1975-02-08,2026-01-25,18479
28759,AW00028759,Melissa,Rivera,United Kingdom,Single,Female,1976-09-30,2026-01-25,18478
28666,AW00028666,Kelvin,Zhu,United States,Married,Male,1979-10-18,2026-01-25,18477
28432,AW00028432,Bruce,Carlson,Germany,Single,Male,1980-12-21,2026-01-25,18476
28165,AW00028165,Alexa,Rivera,Canada,Married,Female,1967-03-09,2026-01-25,18475


In [0]:
%sql
SELECT COUNT(*) FROM store_catalog.gold.dim_customers 

count(1)
18484
